In [ ]:
import requests
import pandas as pd
import numpy as np
from io import BytesIO
import matplotlib.pyplot as plt
import json
import os
from pandas import json_normalize
import networkx as nx
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

from pathlib import Path

# 노트북을 어느 하위 폴더에서 실행해도 저장소 루트를 찾습니다.
_current_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in (_current_dir, *_current_dir.parents)
    if (path / "notebooks").is_dir() and (path / "data").is_dir()
)
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"


In [2]:
import networkx as nx
import pandas as pd

def cal_centrality(original_df):
    large_df = original_df[original_df['amount'] != 0.0]  # 토큰의 전송이 없는 transaction의 경우 제거
    # 네트워크 생성 (유향 그래프)
    G = nx.from_pandas_edgelist(large_df, 'sender_address', 'receiver_address', edge_attr='amount', create_using=nx.DiGraph())  # 거래량으로 가중치를 부여

    # In-degree Centrality
    indegree_centrality = nx.in_degree_centrality(G)
    
    # Out-degree Centrality
    outdegree_centrality = nx.out_degree_centrality(G)
    
    # Betweenness Centrality
    betweenness_centrality = nx.betweenness_centrality(G)
    
    # Closeness Centrality
    closeness_centrality = nx.closeness_centrality(G)
    
    # PageRank (감쇠계수는 보통 0.85)
    pagerank = nx.pagerank(G, alpha=0.85)

    # 중앙성 측정값들을 데이터프레임으로 변환
    centrality = {
        'indegree_centrality': indegree_centrality, 
        'outdegree_centrality': outdegree_centrality, 
        'betweenness_centrality': betweenness_centrality, 
        'closeness_centrality': closeness_centrality, 
        'pagerank': pagerank
    }

    # 중앙성 측정값들을 데이터프레임으로 변환
    df = pd.DataFrame(centrality)
    df.index.name = 'address'
    return df


In [3]:
# 메인 함수
def calculate_daily_centrality(df):
    # timestamp를 datetime 형식으로 변환
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    # 날짜별로 그룹화
    grouped = df.groupby(df['timestamp'].dt.date)
    
    # 결과를 저장할 빈 DataFrame 생성
    result_df = pd.DataFrame()
    
    # tqdm을 사용하여 진행 상황 표시
    for date, group in tqdm(grouped):
        # 해당 날짜의 데이터에 대해 centrality 계산
        daily_centrality = cal_centrality(group)
        
        # 날짜 정보 추가
        daily_centrality['date'] = date
        
        # 결과를 메인 DataFrame에 추가
        result_df = pd.concat([result_df, daily_centrality], ignore_index=False)
    
    return result_df

In [ ]:
input_path = DATA_DIR / "processed" / "wemix" / "WEMIX_2022-11-24~2023-02-23.csv"


In [ ]:
df = pd.read_csv(input_path)


In [ ]:
df.head()


In [8]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df[df['timestamp'] <= datetime(2023,2,23)]

In [10]:
# 함수 사용
result = calculate_daily_centrality(df)

100%|█████████████████████████████████████████| 122/122 [17:16<00:00,  8.50s/it]


In [11]:
result

,indegree_centrality,outdegree_centrality,betweenness_centrality,closeness_centrality,pagerank,date
0x75da6ae2c46b93565ac65b6a299ef055d437fd82,0.000000,0.000124,0.000000,0.000000,0.000049,2022-10-24
0xc6a2ad8cc6e4a7e08fc37cc5954be07d499e7654,0.115475,0.082994,0.090618,0.133950,0.059237,2022-10-24
0x917eed7ae9e7d3b0d875dd393af93fff3fc301f8,0.001853,0.000865,0.004434,0.097496,0.001144,2022-10-24
0x278b83fc5b5f66966e9a1afbdbf8da1f797e0d12,0.000494,0.000124,0.000052,0.076511,0.000454,2022-10-24
0x298b036f0e7f5bec69f3134ed36d87450c16c20d,0.000000,0.000124,0.000000,0.000000,0.000049,2022-10-24
...,...,...,...,...,...,...
0x267caada1de35259ac5842f4d54a9438122dc415,0.000650,0.000000,0.000000,0.089301,0.000199,2023-02-22
0xa8be5d69f5229249d64a8b7f3cc4032464639a60,0.000650,0.000000,0.000000,0.100342,0.000241,2023-02-22
0x8b37af9d981599639c76a489c1009ebfafab88d0,0.000000,0.000650,0.000000,0.000000,0.000197,2023-02-22
0x45d492af8e39af0dc3e20c7eca6c422d4053bd13,0.000650,0.000000,0.000000,0.089301,0.000199,2023-02-22


In [ ]:
output_path = RESULTS_DIR / "wemix" / "network_metrics" / "WEMIX_2022-11-24_to_2023-02-23_centrality.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(output_path)
